# 90 — DNS setup for audit-agent.ca

Points the apex and `www` at GitHub Pages via the GoDaddy Domains API (v3).

**Not a pipeline stage.** Numbered outside the 01–06 range because it is a one-time operations task, not part of the benchmark build.

## How this notebook is safe to run

1. It reads the zone and shows you everything currently there.
2. It computes a plan — records to delete, records to create — and prints it **without applying anything**.
3. Applying requires you to flip `CONFIRM = True` by hand and re-run.

It only touches three `(name, type)` pairs: `@/A`, `@/AAAA`, and `www/CNAME`. Any MX, TXT, or other records in the zone are listed and left alone — a mail record is not collateral damage in a web-hosting change.

The v3 API has no bulk-replace endpoint, so changes are individual creates and deletes. GoDaddy-managed `SOA` and `NS` records cannot be deleted and are never in scope here.

**The token is never in this notebook.** It is read at runtime from Colab Secrets.

In [ ]:
# --- Config. Every tunable value in this notebook lives in this cell. ---
DOMAIN = 'audit-agent.ca'
GITHUB_USER = 'ajrngn'  # CNAME target is ajrngn.github.io

# GitHub Pages apex addresses. Re-check GitHub's "Managing a custom domain"
# doc before a first setup — these change rarely, but they do change.
PAGES_A = ['185.199.108.153', '185.199.109.153',
           '185.199.110.153', '185.199.111.153']
PAGES_AAAA = ['2606:50c0:8000::153', '2606:50c0:8001::153',
              '2606:50c0:8002::153', '2606:50c0:8003::153']

# GoDaddy's v3 API rejects a TTL below 600.
TTL = 600

# Flip to True only after reading the printed plan.
CONFIRM = False

In [ ]:
import requests
from google.colab import userdata

assert GITHUB_USER, 'Set GITHUB_USER — the www CNAME target depends on it'

# Token comes from Colab Secrets (key icon in the sidebar), never a cell literal.
PAT = userdata.get('GODADDY_PAT')

BASE = 'https://api.godaddy.com/v3/domains'
HEADERS = {'Authorization': f'Bearer {PAT}', 'Accept': 'application/json'}

# Only these (name, type) pairs are managed by this notebook. Everything else
# in the zone is reported and left untouched.
MANAGED = {('@', 'A'), ('@', 'AAAA'), ('www', 'CNAME')}


def list_records():
    """All DNS records for the zone, paging through the API."""
    records, page = [], 1
    while True:
        resp = requests.get(f'{BASE}/zones/{DOMAIN}/dns-records',
                            headers=HEADERS,
                            params={'page': page, 'pageSize': 100},
                            timeout=30)
        resp.raise_for_status()
        items = resp.json().get('items', [])
        records.extend(items)
        if len(items) < 100:
            return records
        page += 1


def show(records, title):
    print(f'{title} ({len(records)})')
    for r in sorted(records, key=lambda x: (x['type'], x['name'], x['data'])):
        print(f"  {r['type']:<6} {r['name']:<8} {r['data']:<40} ttl={r.get('ttl')}")

In [ ]:
# Read the zone as it stands today. Nothing is modified by this cell.
current = list_records()

in_scope = [r for r in current if (r['name'], r['type']) in MANAGED]
out_of_scope = [r for r in current if (r['name'], r['type']) not in MANAGED]

show(current, f'Current zone for {DOMAIN}')
print()
show(out_of_scope, 'Out of scope — will NOT be touched')

In [ ]:
# Compute the plan. Still nothing applied.
desired = (
    [{'name': '@', 'type': 'A', 'data': ip, 'ttl': TTL} for ip in PAGES_A]
    + [{'name': '@', 'type': 'AAAA', 'data': ip, 'ttl': TTL} for ip in PAGES_AAAA]
    + [{'name': 'www', 'type': 'CNAME', 'data': f'{GITHUB_USER}.github.io', 'ttl': TTL}]
)

def key(r):
    return (r['name'], r['type'], r['data'].rstrip('.'))

desired_keys = {key(d) for d in desired}
current_keys = {key(r) for r in in_scope}

# Anything in scope that is not wanted — GoDaddy parking records land here.
to_delete = [r for r in in_scope if key(r) not in desired_keys]
to_create = [d for d in desired if key(d) not in current_keys]

print('PLAN')
print()
show(to_delete, '  DELETE')
print()
show(to_create, '  CREATE')
print()
print(f'  unchanged: {len(in_scope) - len(to_delete)} in-scope records already correct')
print(f'  untouched: {len(out_of_scope)} out-of-scope records')
print()
if not to_delete and not to_create:
    print('Zone already matches the target. Nothing to do.')
else:
    print('Review the above, then set CONFIRM = True and run the next cell.')

In [ ]:
# Apply. Deletes first so a replaced apex record does not collide with its
# successor, then creates.
if not CONFIRM:
    raise SystemExit('CONFIRM is False — nothing applied. Read the plan above first.')

for r in to_delete:
    resp = requests.delete(f"{BASE}/zones/{DOMAIN}/dns-records/{r['recordId']}",
                           headers=HEADERS, timeout=30)
    if resp.status_code == 409:
        # SOA and NS are GoDaddy-managed and cannot be removed.
        print(f"  skip   {r['type']:<6} {r['name']:<8} {r['data']}  (managed record)")
        continue
    resp.raise_for_status()
    print(f"  delete {r['type']:<6} {r['name']:<8} {r['data']}")

for d in to_create:
    resp = requests.post(f'{BASE}/zones/{DOMAIN}/dns-records',
                         headers=HEADERS, json=d, timeout=30)
    resp.raise_for_status()
    print(f"  create {d['type']:<6} {d['name']:<8} {d['data']}")

print()
print('Applied.')

In [ ]:
# Verify against the API, then independently against public DNS.
show(list_records(), f'Zone after change — {DOMAIN}')

print()
print('Now confirm public resolution (propagation on a 600 TTL is usually minutes,')
print('but GoDaddy can take an hour — do not re-edit while waiting):')
print(f'  dig +short {DOMAIN} A')
print(f'  dig +short www.{DOMAIN} CNAME')
print()
print('Then in the GitHub repo: Settings > Pages > Custom domain > ' + DOMAIN)
print('and tick Enforce HTTPS once the certificate provisions.')